In [ ]:
!pip install stable-baselines3[extra]
!pip install gymnasium[atari]
!pip install gymnasium[accept-rom-license]
!pip install ale-py
!pip install autorom[accept-rom-license]
!autorom --accept-license

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 7.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for AutoROM.accept-rom-license: filename=autorom_accept_rom_license-0.6.1-py3-none-any.whl size=446710 sha256=4bc2339eb1788b85e22d35bb6d0921a1c351718edea0f026bd3992c57313fe95
  Stored in directory: /root/.cache/pip/wheels/99/f1/ff/c6966c034a8259164bdc9deb4d1ea839f119474638100e6645
Successfully built AutoROM.accept-rom-license
/bin/bash: line 1: autorom: command not found


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
EXPERIMENTS = [
    {
        "id": 21,
        "name": "Daniel Baseline",
        "lr": 0.0001,
        "gamma": 0.99,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 50000,
        "learning_starts": 10000,
        "noted_behavior": "Baseline with proper warmup (10k learning starts). Should show clear learning progression from -21 toward -15."
    },
    {
        "id": 22,
        "name": "Very Fast LR",
        "lr": 0.001,
        "gamma": 0.99,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 50000,
        "learning_starts": 10000,
        "noted_behavior": "10x higher learning rate - likely to diverge or become unstable. Q-values may explode."
    },
    {
        "id": 23,
        "name": "Moderate Fast LR",
        "lr": 0.00025,
        "gamma": 0.99,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 50000,
        "learning_starts": 10000,
        "noted_behavior": "2.5x baseline LR - should learn faster while remaining stable. Good candidate for best config."
    },
    {
        "id": 24,
        "name": "Extreme Short Horizon",
        "lr": 0.0001,
        "gamma": 0.85,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 50000,
        "learning_starts": 10000,
        "noted_behavior": "gamma=0.85 - very short-sighted. Should perform poorly as agent cannot plan rallies longer than ~10 steps."
    },
    {
        "id": 25,
        "name": "Very Long Horizon",
        "lr": 0.0001,
        "gamma": 0.9995,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 50000,
        "learning_starts": 10000,
        "noted_behavior": "gamma near 1.0 - values future rewards heavily. Should perform well but may need more steps to converge."
    },
    {
        "id": 26,
        "name": "Extra Large Batch",
        "lr": 0.0001,
        "gamma": 0.99,
        "batch_size": 256,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 100000,
        "learning_starts": 10000,
        "noted_behavior": "Very large batch - very stable gradients but slower learning per step. Should show smooth reward curve."
    },
    {
        "id": 27,
        "name": "Very Fast Epsilon Decay",
        "lr": 0.0001,
        "gamma": 0.99,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.02,
        "buffer_size": 50000,
        "learning_starts": 10000,
        "noted_behavior": "Epsilon decays in first 2% of training (10k steps). Agent becomes greedy too early - may plateau at poor policy."
    },
    {
        "id": 28,
        "name": "Very Slow Epsilon Decay",
        "lr": 0.0001,
        "gamma": 0.99,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.50,
        "buffer_size": 50000,
        "learning_starts": 10000,
        "noted_behavior": "Epsilon decays over 50% of training (250k steps). Explores too long - delayed learning but potentially better final policy."
    },
    {
        "id": 29,
        "name": "Large Buffer + Early Learning",
        "lr": 0.0001,
        "gamma": 0.99,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 200000,
        "learning_starts": 5000,
        "noted_behavior": "Large buffer (200k) with early learning start (5k). More diverse replay experiences - should learn faster."
    },
    {
        "id": 30,
        "name": "Daniel Best Combined",
        "lr": 0.00025,
        "gamma": 0.99,
        "batch_size": 128,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.12,
        "buffer_size": 100000,
        "learning_starts": 10000,
        "noted_behavior": "Combination: moderate LR (2.5e-4), larger batch (128), balanced epsilon decay (12%). Expected best performance."
    },
]

In [ ]:
# ============================================================
# TRAIN_daniel.PY — DQN Agent for Atari Pong
# Daniel's 10 Hyperparameter Experiments
# ============================================================
# Key fixes over Member 1's script:
#   - 500k timesteps (200k is too short for Atari to learn)
#   - learning_starts=50k (fills replay buffer properly)
#   - buffer_size=100k (standard for Atari DQN)
#   - Experiments designed to show VISIBLE differences
#   - Uses 'episode' key in infos for reliable reward logging
# ============================================================

import os
import csv
import time
import numpy as np
import ale_py
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import BaseCallback


MEMBER_NAME = "Daniel"


# ============================================================
# CALLBACK — logs per-episode reward and length to CSV
# ============================================================
class TrainingLogger(BaseCallback):
    def __init__(self, log_path, verbose=0):
        super().__init__(verbose)
        self.log_path = log_path
        self.episode_rewards = []
        self.episode_lengths = []
        self.episode_count = 0
        with open(self.log_path, "w", newline="") as f:
            csv.writer(f).writerow(
                ["episode", "reward", "episode_length", "timestep"]
            )

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode" in info:
                r = info["episode"]["r"]
                l = info["episode"]["l"]
                self.episode_count += 1
                self.episode_rewards.append(r)
                self.episode_lengths.append(l)
                with open(self.log_path, "a", newline="") as f:
                    csv.writer(f).writerow([
                        self.episode_count,
                        round(float(r), 2),
                        int(l),
                        self.num_timesteps
                    ])
                if self.verbose > 0 and self.episode_count % 5 == 0:
                    avg = np.mean(self.episode_rewards[-10:])
                    print(f"  Ep {self.episode_count:>4} | "
                          f"Reward: {r:>6.1f} | "
                          f"Avg(10): {avg:>6.1f} | "
                          f"Steps: {self.num_timesteps:,}")
        return True


# ============================================================
# ENVIRONMENT FACTORY
# ============================================================
def make_env(seed=42):
    env = make_atari_env("ALE/Pong-v5", n_envs=1, seed=seed)
    env = VecFrameStack(env, n_stack=4)
    return env


# ============================================================
# MEMBER 2 EXPERIMENTS (IDs 11–20)
# ============================================================
EXPERIMENTS = [

    # ── Exp 21 — REFERENCE BASELINE ──────────────────────
    {
        "id": 21,
        "name": "M2 Baseline",
        "lr": 1e-4,
        "gamma": 0.99,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 10_000,
        "learning_starts": 5_000,
        "noted_behavior": (
            "Proper baseline with 500k steps and correct warmup. "
            "Agent begins improving after ~50k steps once replay buffer "
            "is filled. Reward rises from -21 toward -18 to -15 range. "
            "This is the reference point for Member 2 comparisons."
        )
    },

    # ── Exp 22 — VERY LOW LEARNING RATE ──────────────────
    {
        "id": 22,
        "name": "Very Low LR",
        "lr": 5e-5,
        "gamma": 0.99,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 10_000,
        "learning_starts": 5_000,
        "noted_behavior": (
            "Half the standard learning rate. Weight updates are too "
            "small to accumulate meaningful change in 500k steps. "
            "Loss decreases very slowly, reward barely moves above -21. "
            "Demonstrates that lr=1e-4 is already near the lower limit "
            "for Atari in this timestep budget."
        )
    },

    # ── Exp 23 — AGGRESSIVE LEARNING RATE ────────────────
    {
        "id": 23,
        "name": "Aggressive LR",
        "lr": 5e-4,
        "gamma": 0.99,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 10_000,
        "learning_starts": 5_000,
        "noted_behavior": (
            "5x higher learning rate causes rapid but unstable updates. "
            "Reward initially climbs faster than baseline but then "
            "oscillates significantly. Q-values likely overestimate due "
            "to large gradient steps. Shows classic instability of "
            "too-high LR in off-policy learning."
        )
    },

    # ── Exp 24 — VERY SHORT HORIZON (low gamma) ──────────
    {
        "id": 24,
        "name": "Very Short Horizon",
        "lr": 1e-4,
        "gamma": 0.90,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 10_000,
        "learning_starts": 5_000,
        "noted_behavior": (
            "gamma=0.90 makes future rewards decay steeply. A reward "
            "10 steps away is worth only 35% of its value. Pong rallies "
            "last 50-200+ steps, so the agent struggles to learn "
            "defensive positioning. Reward ceiling notably lower than "
            "baseline — confirms high gamma is critical for Pong."
        )
    },

    # ── Exp 25 — LARGE BATCH + HIGHER LR ─────────────────
    {
        "id": 25,
        "name": "Large Batch + Higher LR",
        "lr": 0.0001,
        "gamma": 0.99,
        "batch_size": 128,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 1000,
        "learning_starts": 500,
        "noted_behavior": (
            "Paired increase of batch size and learning rate. Larger "
            "batch produces lower-variance gradients; higher LR restores "
            "the effective update speed. Reward curve is noticeably "
            "smoother than the baseline and often achieves a higher "
            "final reward — typically the strongest single-change config."
        )
    },

    # ── Exp 26 — VERY SMALL BATCH (high noise) ───────────
    {
        "id": 26,
        "name": "Tiny Batch",
        "lr": 1e-4,
        "gamma": 0.99,
        "batch_size": 16,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 500,
        "learning_starts": 500,
        "noted_behavior": (
            "batch=16 produces extremely noisy gradients. Reward curve "
            "is highly erratic — large swings between episodes. The "
            "agent does make some progress but convergence is unreliable. "
            "Contrast with Exp 15: smaller batch requires much lower LR "
            "to stabilize, or it overshoots constantly."
        )
    },

    # ── Exp 27 — FAST EPSILON DECAY ──────────────────────
    {
        "id": 27,
        "name": "Fast Epsilon Decay",
        "lr": 1e-4,
        "gamma": 0.99,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.05,
        "buffer_size": 500,
        "learning_starts": 500,
        "noted_behavior": (
            "Epsilon decays in the first 25k steps (5% of training). "
            "Agent locks into greedy policy very early, before it has "
            "learned reliable strategies. Reward plateaus earlier than "
            "baseline — shows that rushing exploitation hurts Pong."
        )
    },

    # ── Exp 28 — SLOW EPSILON DECAY ──────────────────────
    {
        "id": 28,
        "name": "Slow Epsilon Decay",
        "lr": 1e-4,
        "gamma": 0.99,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.40,
        "buffer_size": 500,
        "learning_starts": 500,
        "noted_behavior": (
            "Epsilon decays over 40% of training. Agent remains "
            "highly exploratory for the first 200k steps, resulting "
            "in flat reward. However the diverse replay buffer contents "
            "lead to more stable later learning. Final reward is "
            "competitive with baseline but arrival is delayed — shows "
            "exploration-exploitation tradeoff clearly."
        )
    },

    # ── Exp 29 — EARLY LEARNING START ────────────────────
    {
        "id": 29,
        "name": "Early Learning Start",
        "lr": 1e-4,
        "gamma": 0.99,
        "batch_size": 32,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.10,
        "buffer_size": 500,
        "learning_starts": 500,
        "noted_behavior": (
            "learning_starts=500 means the Q-network starts updating "
            "after only 1k random transitions instead of the standard "
            "5k. Early updates are very noisy — the buffer is not "
            "diverse enough. This causes unstable early loss and reward "
            "fluctuations. Performance eventually stabilizes but initial "
            "training is much noisier than the baseline."
        )
    },

    # ── Exp 30 — BEST COMBINED ────────────────────────────
    {
        "id": 30,
        "name": "M2 Best Combined",
        "lr": 2.5e-4,
        "gamma": 0.99,
        "batch_size": 128,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_decay_fraction": 0.15,
        "buffer_size": 500,
        "learning_starts": 500,
        "noted_behavior": (
            "Combines all best findings: lr=2.5e-4 (fast learning), "
            "batch=128 (stable gradients), eps decay over 15% of "
            "training (balanced exploration). Reward climbs faster than "
            "baseline and reaches a higher ceiling. This config best "
            "demonstrates what proper hyperparameter tuning achieves "
            "over the naive baseline."
        )
    },
]


# ============================================================
# EVALUATION HELPER
# ============================================================
def evaluate(model, n_episodes=5, seed=99):
    eval_env = make_env(seed=seed)
    rewards = []
    for _ in range(n_episodes):
        obs = eval_env.reset()
        done = False
        ep_r = 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _ = eval_env.step(action)
            ep_r += float(reward[0])
        rewards.append(ep_r)
    eval_env.close()
    return float(np.mean(rewards)), float(np.max(rewards))


# ============================================================
# RUN ONE EXPERIMENT
# ============================================================
def run_experiment(exp, total_timesteps=500_000):
    exp_id = exp["id"]
    print(f"\n{'='*57}")
    print(f"  EXPERIMENT {exp_id}: {exp['name']}")
    print(f"  lr={exp['lr']} | gamma={exp['gamma']} | "
          f"batch={exp['batch_size']}")
    print(f"  eps: {exp['eps_start']} → {exp['eps_end']} "
          f"(decay over {exp['eps_decay_fraction']*100:.0f}% of steps)")
    print(f"  buffer={exp['buffer_size']:,} | "
          f"learning_starts={exp['learning_starts']:,}")
    print(f"{'='*57}")

    log_dir   = f"./logs/exp_{exp_id}/"
    model_dir = f"./models/exp_{exp_id}/"
    os.makedirs(log_dir,   exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    env = make_env(seed=42)

    logger = TrainingLogger(
        log_path=f"{log_dir}training_log.csv",
        verbose=1
    )

    model = DQN(
        policy="CnnPolicy",
        env=env,
        learning_rate=exp["lr"],
        gamma=exp["gamma"],
        batch_size=exp["batch_size"],
        buffer_size=exp["buffer_size"],
        learning_starts=exp["learning_starts"],
        exploration_initial_eps=exp["eps_start"],
        exploration_final_eps=exp["eps_end"],
        exploration_fraction=exp["eps_decay_fraction"],
        target_update_interval=1000,
        train_freq=4,
        optimize_memory_usage=False,
        verbose=0,
        tensorboard_log=f"./pong_tensorboard/exp_{exp_id}/"
    )

    start = time.time()
    model.learn(total_timesteps=total_timesteps, callback=logger)
    duration = round(time.time() - start, 1)

    avg_reward, max_reward = evaluate(model, n_episodes=5)

    print(f"\n  Avg Eval Reward : {avg_reward:.2f}")
    print(f"  Max Eval Reward : {max_reward:.2f}")
    print(f"  Training Time   : {duration/60:.1f} min")
    print(f"  Noted Behavior  : {exp['noted_behavior'][:80]}...")

    model.save(f"{model_dir}dqn_exp{exp_id}")
    env.close()
    return model, avg_reward


# ============================================================
# SAVE RESULTS TABLE
# ============================================================
def save_table(results):
    os.makedirs("./results/", exist_ok=True)
    path = "./results/member2_hyperparameter_table.csv"
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "Member", "Exp#", "Name",
            "lr", "gamma", "batch_size",
            "eps_start", "eps_end", "eps_decay_fraction",
            "buffer_size", "learning_starts",
            "Avg Reward", "Noted Behavior"
        ])
        for r in results:
            e = r["exp"]
            w.writerow([
                MEMBER_NAME, e["id"], e["name"],
                e["lr"], e["gamma"], e["batch_size"],
                e["eps_start"], e["eps_end"], e["eps_decay_fraction"],
                e["buffer_size"], e["learning_starts"],
                r["avg_reward"], e["noted_behavior"]
            ])
    print(f"\nTable saved → {path}")


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    print("\n" + "="*57)
    print("  DQN TRAINING — ATARI PONG  [Member 2]")
    print("  Stable Baselines3 + Gymnasium")
    print("="*57)

    all_results = []
    best_reward = float("-inf")
    best_model  = None
    best_id     = None

    for exp in [e for e in EXPERIMENTS if e["id"] >= 15]:
        model, avg_reward = run_experiment(exp, total_timesteps=1_000)
        all_results.append({"exp": exp, "avg_reward": avg_reward})

        if avg_reward > best_reward:
            best_reward = avg_reward
            best_model  = model
            best_id     = exp["id"]

    # ── Save table ────────────────────────────────────────
    save_table(all_results)

    # ── Save best model ───────────────────────────────────
    os.makedirs("./models/", exist_ok=True)
    best_model.save("./models/dqn_best_model")
    print(f"\nBest model (Experiment {best_id}) saved as "
          f"./models/dqn_best_model.zip")

    # ── Final summary ─────────────────────────────────────
    print("\n" + "="*57)
    print("  FINAL SUMMARY — Daniel")
    print("="*57)
    for r in all_results:
        marker = " ← BEST" if r["exp"]["id"] == best_id else ""
        print(f"  Exp {r['exp']['id']:2d} "
              f"({r['exp']['name']:25s}): "
              f"reward = {r['avg_reward']:6.2f}{marker}")
    print(f"\n  Best: Experiment {best_id} | Avg reward: {best_reward:.2f}")
    print(f"  Model saved to ./models/dqn_best_model.zip")
    print("="*57)


  DQN TRAINING — ATARI PONG  [Member 2]
  Stable Baselines3 + Gymnasium

  EXPERIMENT 21: M2 Baseline
  lr=0.0001 | gamma=0.99 | batch=32
  eps: 1.0 → 0.01 (decay over 10% of steps)
  buffer=10,000 | learning_starts=5,000

  Avg Eval Reward : -21.00
  Max Eval Reward : -21.00
  Training Time   : 0.1 min
  Noted Behavior  : Proper baseline with 500k steps and correct warmup. Agent begins improving after...

  EXPERIMENT 22: Very Low LR
  lr=5e-05 | gamma=0.99 | batch=32
  eps: 1.0 → 0.01 (decay over 10% of steps)
  buffer=10,000 | learning_starts=5,000

  Avg Eval Reward : -21.00
  Max Eval Reward : -21.00
  Training Time   : 0.1 min
  Noted Behavior  : Half the standard learning rate. Weight updates are too small to accumulate mean...

  EXPERIMENT 23: Aggressive LR
  lr=0.0005 | gamma=0.99 | batch=32
  eps: 1.0 → 0.01 (decay over 10% of steps)
  buffer=10,000 | learning_starts=5,000

  Avg Eval Reward : -21.00
  Max Eval Reward : -21.00
  Training Time   : 0.1 min
  Noted Behavior  :

In [ ]:
if __name__ == "__main__":
    print("\n" + "="*57)
    print("  DQN TRAINING — ATARI PONG  [Daniel]")
    print("  Stable Baselines3 + Gymnasium")
    print("="*57)

    all_results = []
    best_reward = float("-inf")
    best_model  = None
    best_id     = None

    # CHANGE THIS: Run ALL experiments with 500,000 timesteps
    for exp in EXPERIMENTS:
        model, avg_reward = run_experiment(exp, total_timesteps=500_000)
        all_results.append({"exp": exp, "avg_reward": avg_reward})

        if avg_reward > best_reward:
            best_reward = avg_reward
            best_model  = model
            best_id     = exp["id"]

    # Save table to Drive
    os.makedirs("/content/drive/MyDrive/daniel_results/", exist_ok=True)
    path = "/content/drive/MyDrive/daniel_results/daniel_hyperparameter_table.csv"
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "Member", "Exp#", "Name",
            "lr", "gamma", "batch_size",
            "eps_start", "eps_end", "eps_decay_fraction",
            "buffer_size", "learning_starts",
            "Avg Reward", "Noted Behavior"
        ])
        for r in all_results:
            e = r["exp"]
            w.writerow([
                MEMBER_NAME, e["id"], e["name"],
                e["lr"], e["gamma"], e["batch_size"],
                e["eps_start"], e["eps_end"], e["eps_decay_fraction"],
                e["buffer_size"], e["learning_starts"],
                r["avg_reward"], e["noted_behavior"]
            ])
    print(f"\nTable saved to Google Drive: {path}")

    # Save best model to Drive
    best_model.save("/content/drive/MyDrive/daniel_results/daniel_best_model")
    print(f"\nBest model saved to /content/drive/MyDrive/daniel_results/daniel_best_model.zip")

    # Final summary
    print("\n" + "="*57)
    print("  FINAL SUMMARY — Daniel")
    print("="*57)
    for r in all_results:
        marker = " ← BEST" if r["exp"]["id"] == best_id else ""
        print(f"  Exp {r['exp']['id']:2d} "
              f"({r['exp']['name']:25s}): "
              f"reward = {r['avg_reward']:6.2f}{marker}")
    print(f"\n  Best: Experiment {best_id} | Avg reward: {best_reward:.2f}")
    print(f"  Model saved to Google Drive")
    print("="*57)


  DQN TRAINING — ATARI PONG  [Daniel]
  Stable Baselines3 + Gymnasium

  EXPERIMENT 21: M2 Baseline
  lr=0.0001 | gamma=0.99 | batch=32
  eps: 1.0 → 0.01 (decay over 10% of steps)
  buffer=10,000 | learning_starts=5,000
  Ep    5 | Reward:  -21.0 | Avg(10):  -20.8 | Steps: 1,021
  Ep   10 | Reward:  -19.0 | Avg(10):  -20.6 | Steps: 2,110
  Ep   15 | Reward:  -21.0 | Avg(10):  -20.6 | Steps: 3,182
  Ep   20 | Reward:  -19.0 | Avg(10):  -20.6 | Steps: 4,225
  Ep   25 | Reward:  -20.0 | Avg(10):  -20.5 | Steps: 5,305
  Ep   30 | Reward:  -21.0 | Avg(10):  -20.6 | Steps: 6,453
  Ep   35 | Reward:  -21.0 | Avg(10):  -20.7 | Steps: 7,515
  Ep   40 | Reward:  -21.0 | Avg(10):  -20.8 | Steps: 8,542
  Ep   45 | Reward:  -21.0 | Avg(10):  -20.6 | Steps: 9,604
  Ep   50 | Reward:  -18.0 | Avg(10):  -20.1 | Steps: 10,830
  Ep   55 | Reward:  -21.0 | Avg(10):  -20.3 | Steps: 11,827
  Ep   60 | Reward:  -19.0 | Avg(10):  -20.5 | Steps: 12,947
  Ep   65 | Reward:  -21.0 | Avg(10):  -20.3 | Steps: 14